# Phase 5 — Model Registration

Retrieves the model artifact from a finished job and registers it in the Azure ML Model Registry.

**Input:** job name from Phase 3 (or best child job from Phase 4 sweep).

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
PROJECT_NAME    = 'my-project'
MODEL_NAME      = f'{PROJECT_NAME}-model'

# Paste the job name from Phase 3 (or best trial from Phase 4)
BEST_JOB_NAME   = 'PASTE_JOB_NAME_HERE'

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

## Section 1 — Connect

In [ ]:
# ── AUTH ──────────────────────────────────────────────────────────────────────
try:
    credential = DefaultAzureCredential()
    credential.get_token('https://management.azure.com/.default')
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
print(f'✓ Connected to: {ml_client.workspace_name}')

## Section 2 — Register Model

In [ ]:
# ── VERIFY JOB STATUS ─────────────────────────────────────────────────────────
job = ml_client.jobs.get(BEST_JOB_NAME)
print(f'Job   : {job.name}')
print(f'Status: {job.status}')

if job.status != 'Completed':
    print('\nWARNING — job is not Completed. Do not register until it finishes.')

In [ ]:
# ── REGISTER ──────────────────────────────────────────────────────────────────
# azureml://jobs/{job_name}/outputs/artifacts/paths/model/ is the standard
# MLflow output path when autolog is enabled.
model = Model(
    path=f'azureml://jobs/{BEST_JOB_NAME}/outputs/artifacts/paths/model/',
    name=MODEL_NAME,
    description=f'Model trained by job {BEST_JOB_NAME}',
    type=AssetTypes.MLFLOW_MODEL,
)

registered_model = ml_client.models.create_or_update(model)

print(f'✓ Model registered: {registered_model.name}')
print(f'  Version         : {registered_model.version}')
print()
print(f'→ Record version {registered_model.version} in README → Model Version Log')

---
## Phase 5 Gate

- [ ] Model visible in Studio → Models tab
- [ ] Version number recorded in README → Model Version Log

Next: Phase 6 — deployment.